# TMol How-To Guide — Metals and Nucleic Acids

## 3. Metals and Nucleic Acids

Metal ions and nucleic acids both score correctly with no special setup -- this notebook is mostly worked examples confirming that. It assumes the basics from [tmol_how_to_guide.ipynb](tmol_how_to_guide.ipynb) and scoring/relax from [02_scoring_and_optimization.ipynb](02_scoring_and_optimization.ipynb).

### Setup

Wheel install, imports, seeding, and device selection live in [notebook_setup.py](notebook_setup.py).

In [1]:
try:
    import google.colab  # noqa: F401
except ImportError:
    IN_COLAB = False
else:
    IN_COLAB = True

if IN_COLAB:
    from urllib.request import urlretrieve

    urlretrieve(
        "https://raw.githubusercontent.com/uw-ipd/tmol/"
        "master/notebooks/notebook_setup.py",
        "notebook_setup.py",
    )

from notebook_setup import *  # noqa: F401,F403

device, param_db = setup()


device=cpu


### Metals

#### Handled by default

Metal ions (ZN, CA, MG, FE, ...) are ordinary residue types in the default chemical database, not ligands -- detecting a metal ion and building its coordinating bonds needs no `prepare_ligands` flag. A `MetalCoordination` energy term scoring that geometry is included in `beta2016_score_function` by default, at weight 1.0.

#### Example

We use a small curated fixture from TMol's own test suite -- a tetrahedral zinc site from [3KS3](https://www.rcsb.org/structure/3KS3) (carbonic anhydrase II) -- rather than fetching the full deposited structure, since it's already cropped to the site of interest. It also includes a crystallization additive (glycerol), which *is* a ligand needing `prepare_ligands=True`.

In [2]:
from tmol.io import atom_array_from_cif, pose_stack_from_biotite
from tmol.tests.data import data_path

metal_structure = atom_array_from_cif(data_path("metal_fixtures") / "zn_tetrahedral_3ks3.cif.zst")
metal_pose_stack, metal_build_context = pose_stack_from_biotite(
    metal_structure, device, prepare_ligands=True, param_db=param_db, return_context=True
)
print(f"n_poses={metal_pose_stack.n_poses}, max_n_blocks={metal_pose_stack.max_n_blocks}")


HIS is seen only at a chain terminus and N, ND1, NE2 could each carry its other connection; it is read as the conventional backbone through N. A copy of the residue in a chain would settle it.


*** Open Babel Warning  in WriteMolecule
  No 2D or 3D coordinates exist. Stereochemical information will be stored using an Open Babel extension. To generate 2D or 3D coordinates instead use --gen2D or --gen3D.


n_poses=1, max_n_blocks=259


There is no single "report coordination" function; the real pattern is to find the metal's block type (it declares coordination sites via `tmol.database.chemical.site_connections`), then walk `pose_stack.inter_residue_connections64` to see which residue fills each site.

In [3]:
from tmol.database.chemical import site_connections

pbt = metal_pose_stack.packed_block_types
chain_labels = metal_pose_stack.pdb_info.chain_labels[0]
residue_labels = metal_pose_stack.pdb_info.residue_labels[0]
irc = metal_pose_stack.inter_residue_connections64

for block in range(metal_pose_stack.max_n_blocks):
    bt_ind = int(metal_pose_stack.block_type_ind64[0, block])
    if bt_ind < 0:
        continue
    block_type = pbt.active_block_types[bt_ind]
    sites = site_connections(block_type)
    if not sites:
        continue
    print(f"{block_type.name3} at chain {chain_labels[block]} residue {residue_labels[block]}:")
    for site_name in sites:
        connection_index = block_type.connection_to_cidx[site_name]
        partner_block, partner_connection = irc[0, block, connection_index].tolist()
        if partner_block < 0:
            print(f"  {site_name}: open")
            continue
        partner_type = pbt.active_block_types[int(metal_pose_stack.block_type_ind64[0, partner_block])]
        atom_name = partner_type.connections[partner_connection].atom
        print(
            f"  {site_name}: chain {chain_labels[partner_block]} residue "
            f"{residue_labels[partner_block]} ({partner_type.name3}) atom {atom_name}"
        )


ZN at chain A residue 262:
  site1: chain A residue 94 (HIS) atom NE2
  site2: chain A residue 96 (HIS) atom NE2
  site3: chain A residue 119 (HIS) atom ND1
  site4: open


Relax it (using the ligand-extended parameter database, since the glycerol needs it) and read off the `metal_coordination` term.

In [4]:
metal_score_function = beta2016_score_function(device, param_db=metal_build_context.parameter_database)
metal_fold_forest = tmol.FoldForest.reasonable_fold_forest(metal_pose_stack)

torch.manual_seed(SEED)
metal_relaxed = tmol.fast_relax(
    metal_pose_stack.clone(), metal_score_function, PackerPalette(), tmol.CartesianMoveMap(), metal_fold_forest,
    num_repeats=1,
)

metal_scorer = metal_score_function.render_whole_pose_scoring_module(metal_relaxed)
weighted_terms = metal_scorer(metal_relaxed.coords, sum_terms=False, apply_weights=True)
metal_coordination_index = metal_score_function.all_score_types().index(ScoreType.metal_coordination)
print(f"metal_coordination energy: {float(weighted_terms[metal_coordination_index, 0].detach()):.2f}")


metal_coordination energy: 0.28


#### Metal cluster example

Multi-ion cofactors (iron-sulfur clusters, and similar) are supported the same way -- still just `prepare_ligands=True` on load, no cluster-specific API. Here a [2Fe-2S] cluster from a ferredoxin ([6E6R](https://www.rcsb.org/structure/6E6R)), each iron coordinated by two cysteines.

In [5]:
fes_structure = atom_array_from_cif(data_path("metal_fixtures") / "fes_ferredoxin_6e6r.cif.zst")
fes_pose_stack, fes_build_context = pose_stack_from_biotite(
    fes_structure, device, prepare_ligands=True, param_db=param_db, return_context=True
)
fes_score_function = beta2016_score_function(device, param_db=fes_build_context.parameter_database)
fes_scorer = fes_score_function.render_whole_pose_scoring_module(fes_pose_stack)
fes_weighted_terms = fes_scorer(fes_pose_stack.coords, sum_terms=False, apply_weights=True)
print(f"n_poses={fes_pose_stack.n_poses}, max_n_blocks={fes_pose_stack.max_n_blocks}")
print(f"metal_coordination energy: {float(fes_weighted_terms[metal_coordination_index, 0]):.2f}")


n_poses=1, max_n_blocks=57
metal_coordination energy: 37.32


### Nucleic acids

#### Handled by default

DNA and RNA residues are likewise ordinary chemistry in the default database. An `NaTorsion` term scoring their backbone, sugar-pucker, and glycosidic torsions is included in `beta2016_score_function` by default; it's an exact zero on protein-only poses, so mixed protein/NA structures score correctly with no special handling either.

#### Example

[1HDD](https://www.rcsb.org/structure/1HDD), an engrailed homeodomain bound to its DNA operator, gives us both an NA torsion energy and a protein-DNA interface in one structure.

In [6]:
dna_cif_text = fetch_cif("1HDD")
dna_pose_stack = pose_stack_from_cif(io.StringIO(dna_cif_text), device, param_db=param_db)
print(f"n_poses={dna_pose_stack.n_poses}, max_n_blocks={dna_pose_stack.max_n_blocks}")

na_score_function = beta2016_score_function(device, param_db=param_db)
na_fold_forest = tmol.FoldForest.reasonable_fold_forest(dna_pose_stack)

torch.manual_seed(SEED)
dna_relaxed = tmol.fast_relax(
    dna_pose_stack.clone(), na_score_function, PackerPalette(), tmol.CartesianMoveMap(), na_fold_forest,
    num_repeats=1,
)
dna_scorer = na_score_function.render_whole_pose_scoring_module(dna_relaxed)
dna_weighted_terms = dna_scorer(dna_relaxed.coords, sum_terms=False, apply_weights=True)
na_torsion_index = na_score_function.all_score_types().index(ScoreType.na_torsion)
print(f"na_torsion energy: {float(dna_weighted_terms[na_torsion_index, 0].detach()):.2f}")


ARG is seen only at a chain terminus and N, NE, NH1, NH2 could each carry its other connection; it is read as the conventional backbone through N. A copy of the residue in a chain would settle it.


n_poses=1, max_n_blocks=156


na_torsion energy: 33.85


The protein-DNA interface energy reuses the block-pair scoring technique from [02_scoring_and_optimization.ipynb](02_scoring_and_optimization.ipynb), with a `polymer_type`-based mask standing in for a chain mask -- selections like this are covered in general in [05_selections.ipynb](05_selections.ipynb).

In [7]:
pbt = dna_relaxed.packed_block_types
protein_mask = torch.zeros((1, dna_relaxed.max_n_blocks), dtype=torch.bool)
dna_mask = torch.zeros((1, dna_relaxed.max_n_blocks), dtype=torch.bool)
for block in range(dna_relaxed.max_n_blocks):
    bt_ind = int(dna_relaxed.block_type_ind64[0, block])
    if bt_ind < 0:
        continue
    polymer_type = pbt.active_block_types[bt_ind].properties.polymer.polymer_type
    protein_mask[0, block] = polymer_type == "amino_acid"
    dna_mask[0, block] = polymer_type == "nucleic_acid"

dna_block_scorer = na_score_function.render_block_pair_scoring_module(dna_relaxed)
dna_block_pair = dna_block_scorer(dna_relaxed.coords)[0].detach()
interface_score = (
    dna_block_pair[protein_mask[0]][:, dna_mask[0]].sum()
    + dna_block_pair[dna_mask[0]][:, protein_mask[0]].sum()
)
print(f"protein blocks: {int(protein_mask.sum())}, DNA blocks: {int(dna_mask.sum())}")
print(f"protein-DNA interface score: {float(interface_score):.2f}")


protein blocks: 114, DNA blocks: 42
protein-DNA interface score: -200.74


The same recipe works unchanged for RNA. [1EHT](https://www.rcsb.org/structure/1EHT) is a theophylline-binding RNA aptamer; here the interesting "other side" is its bound ligand rather than a second polymer chain, covered in [04_ligands_and_ncaas.ipynb](04_ligands_and_ncaas.ipynb), so we report just the NA torsion energy.

In [8]:
rna_cif_text = fetch_cif("1EHT")
rna_pose_stack = pose_stack_from_cif(io.StringIO(rna_cif_text), device, param_db=param_db)
print(f"n_poses={rna_pose_stack.n_poses}, max_n_blocks={rna_pose_stack.max_n_blocks}")

torch.manual_seed(SEED)
rna_relaxed = tmol.fast_relax(
    rna_pose_stack.clone(), na_score_function, PackerPalette(), tmol.CartesianMoveMap(),
    tmol.FoldForest.reasonable_fold_forest(rna_pose_stack), num_repeats=1,
)
rna_scorer = na_score_function.render_whole_pose_scoring_module(rna_relaxed)
rna_weighted_terms = rna_scorer(rna_relaxed.coords, sum_terms=False, apply_weights=True)
print(f"na_torsion energy: {float(rna_weighted_terms[na_torsion_index, 0].detach()):.2f}")


G is seen only at a chain terminus and N1, N2 could each carry its other connection; none of them is a conventional backbone, so it cannot be told apart. A copy of the residue in a chain would settle it.


Unrecognized 3lc TEP


n_poses=1, max_n_blocks=33


na_torsion energy: 63.24
